**`cleanup`**

This notebook demonstrates the automated identification (and optional reclamation) of superfluous intermediate files, as described in `openplaces.io.cleanup`.

Every decision is recomputed directly from disk (combined with per-file tombstone receipts), making it safe to run repeatedly from a notebook, a driver script, or in parallel across cluster jobs. This notebook covers two complementary entry points:

- `op.cleanup`: Walks the recipe dependency DAG rooted at a terminal recipe and finds upstream intermediates whose consumers have all finished.
- `op.compact`: Scans the entire data root and classifies every file as an expected output, reclaimable intermediate, scratch, orphan, or stale receipt.

Both functions default to `dry_run=True` (report only). Nothing in this notebook deletes any data unless you explicitly uncomment a destructive code cell.

# Configure

Set up the recipe and administrative boundaries to clean up.

> [!TIP]
> Setting `admin_ids = None` scans the disk recursively to find all existing administrative unit directories with output files for the specified recipe. This searches all files in the database, including those in obsolete `admin_id` folders.

In [ ]:
import openplaces as op
from openplaces.config import cfg

recipe_id = 'US_footprint-cheer-2026'
admin_ids = ['US-NC-AR']  # Carteret County (pilot)

# cleanup(): Reclaim consumed intermediates for a pipeline run

`op.cleanup` walks the recipe dependency DAG rooted at a terminal recipe to identify upstream outputs with `until_consumed` retention whose consumers have *all* finished (see Section 4.2 of the cleanup design). By default, `dry_run=True` is set, meaning no files will be deleted until you explicitly pass `dry_run=False`.

In [ ]:
report = op.cleanup(recipe_id, admin_ids=admin_ids, verbose=True)
report

In [ ]:
report['action'].value_counts()

In [ ]:
# What would be reclaimed
report[report['action'] == 'would_delete'].sort_values('size_mb', ascending=False)

In [ ]:
# What's being kept, and why
report[report['action'] == 'blocked'][
    ['path', 'recipe_id', 'admin_id', 'size_mb', 'blocked_by']
]

## Executing deletion

Actual deletion requires passing `dry_run=False` explicitly. Each deleted output leaves a tombstone receipt (`<name>.consumed.json`) next to its original path. This allows subsequent runs to skip re-ingesting a deliberately deleted output without treating it as missing.

Useful parameters:
- `stages=('ingest',)`: Only reclaim outputs from specific pipeline stages.
- `include_images=True`: Also consider per-building image caches (off by default, as regenerating them requires a paid re-fetch).
- `aggressive=True`: Additionally treat `core` outputs as reclaimable once the curated dataset exists (note: `enrich` evidence is always kept since source images may no longer be available).

Uncomment the cell below to perform the actual cleanup.

In [ ]:
# report = op.cleanup(recipe_id, admin_ids=admin_ids, dry_run=False, verbose=True)
# report

# compact(): Identify superfluous files across the entire data root

`op.compact` acts as a broader garbage collector. It performs an "existent-first" scan of every file within the specified buckets (`cache`, `heap`, `external`, `core`, and `out` by default), matching them against the current recipe tree to classify them:

- `final`: Expected output with a retention of `keep` (reported only).
- `intermediate/needed`: An `until_consumed` file where at least one consumer is still incomplete.
- `intermediate/consumed`: An `until_consumed` file where all consumers are complete (reclaimable).
- `heap`: Freshly unzipped scratch space (always reclaimable).
- `orphan`: Files that match no recipe in the current tree (e.g., from renamed/removed recipes or stray files).
- `receipt/stale`: A tombstone receipt whose recorded consumers have all vanished.

Because deleting orphans is a high-risk operation, it has additional guards:
1. A file must be older than `orphan_min_age_days` (defaults to 14).
2. Shared buckets (`external`, `share`, `raw`) are protected unless `include_shared=True` is set.
3. The recipe tree must have successfully loaded at least 25 recipes (preventing a misconfigured recipe path from classifying all files as orphans).

In [ ]:
report = op.compact()
report

In [ ]:
report.groupby('class')['size_mb'].sum().sort_values(ascending=False)

In [ ]:
# Concrete "superfluous file" candidates: match no recipe in the current tree
import pandas as pd

report[report['class'].eq('orphan')].sort_values('size_mb', ascending=False)

In [ ]:
pd.set_option('display.max_colwidth', 255)
pd.set_option('display.max_rows', 100)
pd.set_option('display.min_rows', 100)
report[report['class'].eq('orphan') & report['bucket'].ne('external')]['path']

In [ ]:
# Tombstone receipts whose recorded consumers have all disappeared
report[report['class'] == 'receipt/stale']

## Executing reclamation

Reclamation requires two explicit actions: setting `dry_run=False` *and* providing a non-empty `delete` tuple selected from `{'consumed', 'heap', 'orphans'}`. Orphans are kept separate from `consumed` and `heap` as they carry the highest risk of accidental deletion.

Uncomment the cells below to perform the actual reclamation.

In [ ]:
# op.compact(delete=('consumed', 'heap'), dry_run=False)

# Orphans separately, since they carry the most risk:
# op.compact(delete=('orphans',), dry_run=False)

In [ ]:
report = op.compact(delete=('orphans',), buckets=('core', 'cache'))

report

In [ ]:
report['class'].value_counts()

# Configuration

Both cleanup functions read their settings from the `retention.cleanup` section of the user configuration:

- `enabled`: The master switch for the stage-level `cleanup_consumed_inputs` hook.
- `honor_receipts`: Dictates whether a tombstone receipt can be used to skip regenerating an already-deleted output.
- `include_images`: The default opt-in setting for image caches.
- `exclude_patterns`: Glob patterns (relative to `data_root`) of files that `compact` should never touch, e.g., `['**/backups/**']`.

The `share` and `raw` buckets serve as a hard floor below the configuration layer; files within these buckets can never be marked as deletable by any configuration.

In [ ]:
(cfg.get('retention') or {}).get('cleanup') or {}

# Receipts

Tombstone receipts record what was deleted, when, and which consumer outputs justified the deletion. This evidence allows subsequent runs to distinguish a file that was "deleted on purpose" from one that was "never generated."

In [ ]:
# Inspect the receipt for a path that compact() reported as 'intermediate/consumed',
# or that a prior real (dry_run=False) run of this notebook reclaimed:
# from openplaces.io.cleanup import read_receipt
# read_receipt(cfg.data_root / '<relative path from a report row>')